# Discovering Music Listening Patterns Using Frequent Itemset Mining
### An Association-Rule-Based Artist Recommender on the Last.fm (HetRec 2011) Dataset

> **📌 OUTLINE / GUIDE NOTEBOOK.** Each **GUIDE** cell lists what the section must contain. It follows the BDCC Lab 1 instructions, your approved game plan, and the class FIM materials (*Frequent Itemset Mining* notebook and the Sessions 3–4 slides). Replace the guide bullets with your write-up, fill in the `TODO` cells, and delete these notes before submitting.

**GUIDE: Title block**
- Keep the game-plan title (above) and a one-line subtitle naming the method: FIM → association rules → recommender.
- ⚠️ **No names, student numbers, or anything that identifies you** (lab rule). Use your SLT label only if allowed.

## Submission Checklist (delete before submitting)
- [ ] The notebook **regenerates the whole report when run top to bottom** (*Kernel → Restart & Run All*).
- [ ] Submit the `.ipynb` **and its HTML export**.
- [ ] Supplementary notebooks / `.py` files (e.g., `fim_utils.py` with your own Apriori/FP-Growth/`assoc` functions) are **referred to in this main notebook** and included.
- [ ] All relevant files, **including images** (e.g., pipeline diagram), are included.
- [ ] **No raw, preprocessed, or processed data files** are included unless a visualization needs one. Read data straight from the shared folder on Jojie.
- [ ] **File paths do not include your username.**
- [ ] No names or identifying info anywhere (cells, file names, plot titles, notebook metadata).
- [ ] The recommender has a **performance evaluation** (lab requirement).
- [ ] Readable for **peer review**: every table and chart has a caption and an interpretation.

## Abstract

**GUIDE**
- 150–250 words covering problem → data → method → key findings → implication. **Write it last.**
- Data: Last.fm HetRec 2011, 1,892 users (transactions), 17,632 artists (items), ~92.8K user–artist records.
- Method: users as transactions and artists as items; Apriori and FP-Growth (pyFIM) at a chosen *minsup*; association rules filtered by *minconf* and ranked by lift; rule-based recommender.
- 1–2 headline numbers: e.g., # frequent itemsets at your minsup, one standout rule with its confidence and lift, recommender hit rate / precision@k vs. the popularity baseline.

## Table of Contents
1. Introduction
2. Problem Statement
3. Motivation
4. Data Source and Description
5. Methodology
6. Data Loading
7. Exploratory Data Analysis
8. Preprocessing: Building the Transaction Database
9. Frequent Itemset Mining
10. Association Pattern Mining
11. Recommender System from Association Rules
12. Performance Evaluation
13. Results and Discussion
14. Conclusion
15. Recommendations
16. References

## 1. Introduction

Streaming platforms compile listening histories by tracking their users' listening activity at large scale. Listening histories show not only the user’s preference towards particular artists, but also their preference towards certain clusters of artists that share similar genres, eras, voice qualities, and cultures. Last.fm is an online music platform that captures such behavior through “scrobbling”, which records the artists and songs a user plays to build a profile of their listening behavior (Last.fm, n.d.). The HetRec 2011 Last.fm dataset used in this report includes the listening records of 1,892 users, forming a good foundation for analyzing co-listening relationships between 17,632 artists (Cantador et al., 2011; GroupLens, 2013).

Frequent itemset mining (FIM) and association pattern mining were initially developed for market basket analysis, wherein companies tried to identify items commonly purchased by their customers (Agrawal & Srikant, 1994). In that scenario, each customer’s purchase basket becomes the transaction, and each product becomes the item. This report follows the same approach in the case of music listening, where the listening history of each individual becomes the basket, and the artists listened to become the items. Thus, FIM discovers frequent sets of artists appearing together in users’ listening histories, and association pattern mining derives the rules of the form “If a user listens to artist A, they also listen to artist B” (Agrawal & Srikant, 1994; Aggarwal, 2015).

As discussed in class, FIM does not end with the identification of frequent itemsets. Frequent itemsets show which combinations of artists occur frequently, while association pattern mining transforms frequent itemsets into meaningful recommendation rules using confidence and lift measures (Agrawal & Srikant, 1994; Fournier-Viger et al., 2017). For instance, an association between Depeche Mode and Martin L. Gore can form the basis for a "fans also like" recommendation. The rule-based technique is especially valuable for discovering new music because it uses only the listening activity of users, without any need for ratings, and it generates recommendations that can be justified with reference to the artists that triggered them. Previous research has also applied association-rule mining techniques like the Apriori algorithm to construct recommendation systems from user–product interaction data (Talwar et al., n.d.).

In this report, we identify artists that are often listened to together by Last.fm users and assess whether these associations can recommend artists that users may also enjoy. The report first introduces the Last.fm dataset and builds the user–artist transaction database (Cantador et al., 2011; GroupLens, 2013). It then mines frequent artist itemsets using the Apriori and FP-Growth algorithms (Agrawal & Srikant, 1994; Han et al., 2000). Next, it generates association rules based on support, confidence, and lift, and uses the highest-quality rules to recommend artists. Finally, it evaluates the recommender against a popularity baseline and discusses the results.


## 2. Problem Statement

**GUIDE**
- Use the game-plan problem: *Which artists frequently occur together in users' listening histories, and can these relationships be used to discover artists a user may also be interested in?*
- Optional sub-questions:
  1. Which artist combinations are frequent (support ≥ *minsup*)?
  2. Which rules $A \rightarrow B$ are reliable (high confidence) **and** genuinely associated (lift > 1)?
  3. How well does a rule-based recommender predict artists a user actually listens to, compared with recommending popular artists?

## 3. Motivation

**GUIDE**
- Who benefits: platforms (discovery, "fans also like", playlists), artists and labels (fan overlap, collaborations, co-marketing, tour line-ups), listeners (finding new music).
- Why FIM and association rules: **interpretable** ("recommended because you listen to X and Y"), need no ratings, only listening presence. This matches the class example of using FIM for recommender systems.

## 4. Data Source and Description

**GUIDE**
- **Source:** HetRec 2011 Last.fm dataset (`hetrec2011-lastfm-2k`), GroupLens: https://grouplens.org/datasets/hetrec-2011/. On Jojie: `hetrec2011/lastfm` (~16 MB).
- **Scope:** 1,892 users · 17,632 artists · 92,834 user–artist records · 11,946 tags · 186,479 tag assignments · 12,717 bi-directional friend relations.
- **File table** (state which files you use and why):

| File | Columns | Description | Used? |
|---|---|---|---|
| `user_artists.dat` | `userID`, `artistID`, `weight` | Artists each user listened to; `weight` = play count | ✅ **Main file**: builds the transaction database |
| `artists.dat` | `id`, `name`, `url`, `pictureURL` | Artist metadata | ✅ Maps IDs → names so itemsets and rules are readable |
| `tags.dat` | `tagID`, `tagValue` | Tag vocabulary (genres, moods…) | Optional: label or interpret itemsets by genre |
| `user_taggedartists.dat` | `userID`, `artistID`, `tagID`, `day`, `month`, `year` | User-assigned artist tags with date | Optional: genre context |
| `user_taggedartists-timestamps.dat` | `userID`, `artistID`, `tagID`, `timestamp` | Same as above, timestamp in ms | Optional (duplicate of above) |
| `user_friends.dat` | `userID`, `friendID` | Friend pairs (each listed both ways) | Probably unused: say so and why |

- **Format:** tab-separated, header row, one record per line. `artists.dat` / `tags.dat` contain non-ASCII names.
- **Data dictionary** for the columns you use (name, type, meaning).
- **Known limitations** to state up front:
  - Each user holds only their **top ~50 artists** (median = max = 50). "Listened to" really means "is in the user's top 50".
  - Data is from ~2011. The most common artists are Lady Gaga (32% of users), Britney Spears, Rihanna, The Beatles and Katy Perry.
- **License:** non-commercial use; cite Last.fm and HetRec'11.

## 5. Methodology

**GUIDE**: Describe the pipeline (ideally as a **flowchart image** included in the submission), then define the concepts **using the class notation**.

**Pipeline**
1. Load data from the shared Jojie folder
2. EDA
3. Build the transaction database $\mathscr{T}$ (one transaction per user)
4. Frequent itemset mining with Apriori and FP-Growth
5. Association pattern mining (rules filtered by *minconf*, ranked by lift)
6. Rule-based recommender
7. Hold-out evaluation against a popularity baseline
8. Insights and recommendations

**Definitions to include (from the class notebook):**
- Database $\mathscr{T} = \{T_1, \dots, T_N\}$; each $T_i$ is a set of items from the universe $\mathscr{U}$, $d = |\mathscr{U}|$. Here $N$ = users and $\mathscr{U}$ = artists. Only **presence** counts and play counts are ignored, as in the class notebook.
- Itemset $X \subseteq \mathscr{U}$; a $k$-itemset has $|X| = k$.
- **Absolute support** (the course's definition of support): $sup(X) = |\{T_i \mid X \subseteq T_i\}|$
- **Relative support:** $relSup(X) = sup(X)/N$. ⚠️ Say which one you report; libraries often mean *relative* support.
- **Frequent itemset:** $sup(X) \geq minsup$.
- **Maximal frequent itemset:** frequent, and no superset is frequent. **Closed itemset:** no superset has the same support.
- **Confidence:** $conf(A \rightarrow B) = \dfrac{sup(A \cup B)}{sup(A)}$, with antecedent $A$ and consequent $B$.
- **Lift:** $lift(A \rightarrow B) = \dfrac{conf(A \rightarrow B)}{relSup(B)}$. Above 1 = positive association, 1 = none, below 1 = negative.

**Algorithms (explain briefly in your own words):**
- **Brute force:** evaluates all $2^d - 1$ itemsets. With $d = 17{,}632$ artists this is infeasible, which is why it is excluded (cite the class runtime argument: exponential in items, linear in transactions).
- **Apriori:** breadth-first search over the itemset lattice using **support monotonicity** and the **downward closure property** to prune candidates.
- **FP-Growth:** pattern growth on a prefix tree using **projected databases**, so it never generates candidates that are absent from the data.
- **Tools:** `pandas`, `numpy`, `scipy.sparse.csr_matrix`, `matplotlib`, and **pyFIM** (`import fim`), the library used in class. Optionally reuse your own `apriori`, `fpgrowth`, and `assoc` functions from the class exercises in a supplementary `fim_utils.py`, and say so here.

## 6. Data Loading

This report reads four of the six files in the HetRec 2011 Last.fm release directly from the shared, read-only copy on Jojie, so no data files are submitted with it (Cantador et al., 2011; GroupLens, 2013). Each file has a specific role:

- `user_artists.dat` holds the user–artist records that become the transaction database.
- `artists.dat` maps each `artistID` to a readable artist name.
- `tags.dat` and `user_taggedartists.dat` hold user-assigned tags, which are used only to describe and interpret artists and itemsets.

`user_friends.dat` is not loaded because the social network is outside the scope of an itemset-based recommender. `user_taggedartists-timestamps.dat` is not loaded because it repeats the same 186,479 tag assignments as `user_taggedartists.dat`, with the date stored as a millisecond timestamp instead of day, month, and year columns.

All files are tab-separated with a header row. `tags.dat` is Latin-1 encoded and raises a decode error when read as UTF-8, so it is read with `encoding='latin-1'`. `artists.dat` is UTF-8 and contains non-Latin names, such as Cyrillic band names.

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.sparse import csr_matrix
import fim  # pyFIM (used in class)

# Optional supplementary module with your class-exercise implementations
# from fim_utils import apriori as my_apriori, fpgrowth as my_fpgrowth, assoc

# One plotting style for every figure in the report
PRIMARY = '#2a78d6'  # data marks
MUTED = '#52514e'    # reference lines and annotations
plt.rcParams.update({
    'figure.figsize': (8, 4.5),
    'figure.dpi': 110,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
    'axes.labelcolor': MUTED,
    'axes.edgecolor': '#b5b4ae',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.color': '#e6e5e0',
    'grid.linewidth': 0.8,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
})

In [ ]:
# Shared read-only copy on Jojie. The local folder holds the same unmodified
# GroupLens release and is used only when running outside Jojie.
JOJIE_DIR = '/mnt/data/public/hetrec2011/lastfm/'  # TODO: confirm on Jojie
LOCAL_DIR = 'hetrec2011-lastfm-2k/'
DATA_DIR = JOJIE_DIR if os.path.isdir(JOJIE_DIR) else LOCAL_DIR

user_artists = pd.read_csv(
    os.path.join(DATA_DIR, 'user_artists.dat'), sep='\t')
artists = pd.read_csv(
    os.path.join(DATA_DIR, 'artists.dat'), sep='\t', encoding='utf-8')
tags = pd.read_csv(
    os.path.join(DATA_DIR, 'tags.dat'), sep='\t', encoding='latin-1')
user_tagged = pd.read_csv(
    os.path.join(DATA_DIR, 'user_taggedartists.dat'), sep='\t')

tables = {
    'user_artists': user_artists,
    'artists': artists,
    'tags': tags,
    'user_taggedartists': user_tagged,
}

In [ ]:
def table_overview(tables):
    """Summarize the size and columns of each loaded table.

    Parameters
    ----------
    tables : dict of str to pandas.DataFrame
        Loaded tables keyed by file name.

    Returns
    -------
    pandas.DataFrame
        One row per table with its row count, column count, and the
        data type of each column.
    """
    rows = []
    for name, df in tables.items():
        rows.append({
            'table': name,
            'rows': f'{len(df):,}',
            'columns': df.shape[1],
            'columns (dtype)': ', '.join(
                f'{col} ({dtype})' for col, dtype in df.dtypes.items()),
        })
    return pd.DataFrame(rows).set_index('table')


display(table_overview(tables))

**Table 1.** Size, columns, and data types of the four loaded tables. `user_artists.dat` has one row per user–artist pair, and `user_taggedartists.dat` has one row per tag assignment.

In [ ]:
for name, df in tables.items():
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]} columns')
    display(df.head(3))

**Table 2.** First three rows of each loaded table, in the order of Table 1. In `user_artists`, `weight` is the number of times the user played the artist.

## 7. Exploratory Data Analysis

This section first checks data quality, then describes the properties that matter most for frequent itemset mining: how many artists each user has, how many users each artist has, and how many artists remain at each candidate *minsup*. Here, the **support** of an artist is its absolute support, the number of transactions (users) that contain it, and its **relative support** divides this by $N$ (Section 5). All statistics use the full dataset; the train/test split is made in Section 8.

### 7.1 Data Quality Checks

In [ ]:
name_key = artists['name'].str.strip().str.lower()
tag_key = tags['tagValue'].str.strip().str.lower()

checks = pd.DataFrame([
    ('user_artists', 'Null values',
     user_artists.isna().sum().sum()),
    ('user_artists', 'Duplicate (userID, artistID) pairs',
     user_artists.duplicated(['userID', 'artistID']).sum()),
    ('user_artists', 'artistID not found in artists.dat',
     (~user_artists['artistID'].isin(artists['id'])).sum()),
    ('user_artists', 'Play count (weight) of zero or less',
     (user_artists['weight'] <= 0).sum()),
    ('artists', 'Null values',
     artists.isna().sum().sum()),
    ('artists', 'Duplicate names (exact)',
     artists['name'].duplicated().sum()),
    ('artists', 'Duplicate names (ignoring case)',
     name_key.duplicated().sum()),
    ('tags', 'Null values',
     tags.isna().sum().sum()),
    ('tags', 'Duplicate tag names (ignoring case)',
     tag_key.duplicated().sum()),
    ('user_taggedartists', 'Null values',
     user_tagged.isna().sum().sum()),
    ('user_taggedartists', 'artistID not found in artists.dat',
     (~user_tagged['artistID'].isin(artists['id'])).sum()),
    ('user_taggedartists', 'tagID not found in tags.dat',
     (~user_tagged['tagID'].isin(tags['tagID'])).sum()),
    ('user_taggedartists', 'Dated before Last.fm launched (2002)',
     (user_tagged['year'] < 2002).sum()),
], columns=['table', 'check', 'count'])

display(checks.set_index(['table', 'check']))
print('Columns with null values:')
for name, df in tables.items():
    nulls = df.isna().sum()
    for col, count in nulls[nulls > 0].items():
        print(f'  {name}.{col}: {count:,}')

**Table 3.** Data quality checks on the four loaded tables. A count of zero means the check passed.

The core table, `user_artists.dat`, is clean. It has no null values, no repeated (user, artist) pair, and no play count of zero or less, and every artist ID has a name in `artists.dat`. Three issues remain in the supporting tables, and each is investigated below before deciding how to handle it:

1. 444 artists have no `pictureURL`. These are the only null values in the data.
2. 12 artist names repeat an earlier name when letter case is ignored (for example, *Слот* and *СЛОТ*), and each variant has its own ID.
3. In `user_taggedartists.dat`, 1,538 tag assignments refer to artists that are not in `artists.dat`, and 5 are dated before Last.fm existed.

In [ ]:
# Support = number of transactions (users) that contain the artist
artist_support = user_artists.groupby('artistID').size()

# Is a missing picture related to how popular the artist is?
picture_status = (
    artists.set_index('id')['pictureURL'].notna()
    .map({True: 'has pictureURL', False: 'pictureURL missing'})
)
display(
    artist_support.groupby(picture_status).describe()
    [['count', 'mean', '50%', 'max']]
    .rename(columns={'50%': 'median'})
)
missing_picture = artists.loc[artists['pictureURL'].isna(), 'name']
print('Examples without a picture:', ', '.join(missing_picture.head(6)))

# How popular are the artists whose names differ only in case?
case_variants = artists.loc[name_key.duplicated(keep=False), 'id']
print(f'\n{len(case_variants)} artist IDs share a name with another ID when '
      f'case is ignored; their highest support is '
      f'{artist_support[case_variants].max()} users.')

# Tagged artists that are missing from artists.dat
orphan_tags = user_tagged[~user_tagged['artistID'].isin(artists['id'])]
print(f"\n{len(orphan_tags):,} tag assignments refer to "
      f"{orphan_tags['artistID'].nunique()} artists not in artists.dat.")
print('Tag assignments dated before 2002 by year:',
      user_tagged.loc[user_tagged['year'] < 2002, 'year']
      .value_counts().sort_index().to_dict())

**Table 4.** Support of artists with and without a `pictureURL`. All 444 artists without a picture appear in at most two transactions.

**How each issue is handled**

- **Missing `pictureURL`.** The missing values are not random. Every one of the 444 artists has a support of 1 or 2, and many names look like one-off entries, such as collaborations or a website name. This suggests that Last.fm had no image for these very obscure entries when the data was collected. The column is display metadata that the analysis never uses, so nothing is imputed: `url` and `pictureURL` are dropped, only `id` and `name` are kept, and no records are removed.
- **Names that differ only in case.** The 23 IDs involved have a support of at most 8 users (0.4% of users), far below every *minsup* considered in Section 7.6, so none of them can appear in a frequent itemset. Merging them would not change any result, so they are kept as separate items. Exact names are unique, so the `artistID` → `name` mapping is one-to-one.
- **Tag anomalies.** Tags are used only to describe artists that appear in transactions, so the 390 tagged artists missing from `artists.dat` drop out when tags are matched to listened artists. Tag dates are never used, so the 5 implausible dates (1956, 1957, and 1979) have no effect.

In [ ]:
# url and pictureURL are display metadata; the analysis needs id -> name
artists = artists[['id', 'name']]
artist_names = artists.set_index('id')['name']

### 7.2 Size and Sparsity

In [ ]:
n_users = user_artists['userID'].nunique()
n_artists = user_artists['artistID'].nunique()
n_records = len(user_artists)

# Binary user x artist matrix: 1 if the artist is in the user's transaction
user_codes, _ = pd.factorize(user_artists['userID'])
artist_codes, _ = pd.factorize(user_artists['artistID'])
listen_matrix = csr_matrix(
    (np.ones(n_records, dtype=np.int8), (user_codes, artist_codes)),
    shape=(n_users, n_artists),
)
density = listen_matrix.nnz / (n_users * n_artists)
sparse_mb = (listen_matrix.data.nbytes + listen_matrix.indices.nbytes
             + listen_matrix.indptr.nbytes) / 1e6
dense_mb = n_users * n_artists / 1e6  # one byte per cell as int8

display(pd.Series({
    'Users, N (transactions)': f'{n_users:,}',
    'Artists, d (items)': f'{n_artists:,}',
    'User-artist records (non-zero cells)': f'{n_records:,}',
    'Density of the N x d matrix': f'{density:.2%}',
    'Dense int8 matrix (MB)': f'{dense_mb:.1f}',
    'Sparse CSR matrix (MB)': f'{sparse_mb:.2f}',
}, name='value').to_frame())

**Table 5.** Size and sparsity of the binary user × artist matrix. Only 0.28% of its cells are non-zero, so the sparse CSR form needs 0.47 MB instead of 33.4 MB for the dense form.

The transaction database has $N$ = 1,892 transactions over $d$ = 17,632 items. A brute-force search would have to evaluate $2^{17{,}632} - 1$ candidate itemsets, which is why the methodology relies on Apriori and FP-Growth (Section 5). The matrix is extremely sparse because each user holds at most 50 of the 17,632 artists, which is why the `csr_matrix` representation from class suits this data.

### 7.3 Transaction Length

In [ ]:
basket_size = user_artists.groupby('userID').size()
n_full = (basket_size == 50).sum()
short = basket_size[basket_size < 50]

fig, ax = plt.subplots()
ax.hist(basket_size, bins=np.arange(0.5, 51.5), color=PRIMARY, rwidth=0.8)
ax.set_yscale('log')
ax.text(48.5, n_full, f'{n_full:,} users ({n_full / n_users:.1%})',
        ha='right', va='center', color=MUTED)
ax.set_xlabel('Artists in the transaction, $n_i = |T_i|$')
ax.set_ylabel('Number of users (log scale)')
ax.set_title('Almost every user has exactly 50 artists')
plt.show()

print(f'Users with fewer than 50 artists: {len(short)} '
      f'({len(short) / n_users:.1%}); median {short.median():.0f}, '
      f'min {short.min()}, {short.sum():,} records in total')

**Figure 1.** Number of artists in each user's transaction ($n_i = |T_i|$), with the y-axis on a log scale. 1,829 of the 1,892 users (96.7%) have exactly 50 artists; the other 63 have between 1 and 49 (median 18).

The dataset stores each user's 50 most-played artists, not their full listening history. A transaction therefore means "is in the user's top 50" rather than "has ever listened to", and its length says almost nothing about how active a user is. Because nearly every transaction has the same length, every user contributes about equally to support counts, and heavy listeners cannot dominate them. The 63 short transactions are kept: they belong to genuine users who listened to fewer artists, and dropping them would remove 3.3% of users for no benefit.

### 7.4 Item Support Distribution

In [ ]:
support_freq = artist_support.value_counts().sort_index()

fig, ax = plt.subplots()
ax.scatter(support_freq.index, support_freq.values, s=16, color=PRIMARY)
ax.set_xscale('log')
ax.set_yscale('log')
for rel_minsup in (0.01, 0.05):
    abs_minsup = int(np.ceil(rel_minsup * n_users))
    ax.axvline(abs_minsup, color=MUTED, linestyle='--', linewidth=1)
    ax.text(abs_minsup * 1.1, support_freq.max(),
            f'minsup {rel_minsup:.0%}\n({abs_minsup} users)',
            va='top', fontsize=9, color=MUTED)
ax.set_xlabel('Support (transactions containing the artist, log scale)')
ax.set_ylabel('Number of artists (log scale)')
ax.set_title('Most artists appear in only one transaction')
plt.show()

for max_support in (1, 2):
    count = (artist_support <= max_support).sum()
    print(f'Artists with support <= {max_support}: {count:,} '
          f'({count / n_artists:.1%})')
print(f'Median support: {artist_support.median():.0f}; '
      f'maximum: {artist_support.max()}')

**Figure 2.** Number of artists at each support value, on log–log axes. The dashed lines mark the absolute support for a relative *minsup* of 1% (19 users) and 5% (95 users). 10,679 artists (60.6%) appear in exactly one transaction.

Artist support has a long tail: the median artist appears in a single transaction, while the most popular appears in 611. Nearly three-quarters of artists (73.8%) appear in at most two transactions, so they cannot be part of a frequent itemset at any useful *minsup*. This has two consequences. First, the mined rules, and the recommender built on them, can only ever recommend a few hundred popular artists, which is a built-in popularity bias. Second, the item universe that FIM actually has to search is far smaller than $d$ = 17,632, which keeps Apriori and FP-Growth tractable.

### 7.5 Most Common Artists

In [ ]:
top_artists = (
    artist_support.nlargest(20)
    .rename(index=artist_names)
    .sort_values()  # ascending, so the largest bar is drawn at the top
)

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(top_artists.index, top_artists.values, height=0.6, color=PRIMARY)
for name, support in top_artists.items():
    ax.text(support + 5, name, f'{support / n_users:.1%}',
            va='center', fontsize=9, color=MUTED)
ax.set_xlim(0, top_artists.max() * 1.12)
ax.grid(axis='y', visible=False)
ax.set_xlabel('Absolute support (users), labeled with relative support')
ax.set_title('Pop acts lead the 20 most common artists')
plt.show()

**Figure 3.** The 20 artists with the highest absolute support, each labeled with its relative support. Lady Gaga appears in 611 transactions (32.3% of users).

Mainstream pop acts of the late 2000s lead the list: 14 of the 20 artists, including the top three, are pop-oriented singers or groups such as Lady Gaga, Britney Spears, and Rihanna. The other six are rock and alternative bands: The Beatles, Muse, Paramore, Radiohead, Coldplay, and The Killers. This mix likely reflects the 2011 snapshot and Last.fm's user base at the time rather than all-time popularity.

Even the most common artist appears in fewer than a third of transactions, so every itemset of two or more artists has a relative support below 32.3%. These artists will also appear as consequents in many rules simply because they are common. A rule that ends in Lady Gaga can reach a high confidence while having a lift close to 1, which is why Section 10 ranks rules by lift rather than by confidence alone.

### 7.6 Candidate Range for *minsup*

In [ ]:
def minsup_summary(rel_minsup):
    """Count the artists and records that survive a support threshold.

    By downward closure, an artist below minsup cannot belong to any
    frequent itemset, so the surviving artists are the only candidates.

    Parameters
    ----------
    rel_minsup : float
        Relative minimum support, as a fraction of users.

    Returns
    -------
    dict
        The threshold in relative and absolute terms, and the number and
        share of artists and records that meet it.
    """
    abs_minsup = int(np.ceil(rel_minsup * n_users))
    kept = artist_support[artist_support >= abs_minsup]
    records_kept = kept.sum()  # one record per (user, kept artist)
    return {
        'relative minsup': rel_minsup,
        'absolute minsup': abs_minsup,
        'artists kept': len(kept),
        '% of artists': len(kept) / n_artists,
        'records kept': records_kept,
        '% of records': records_kept / n_records,
    }


table_grid = [0.01, 0.02, 0.03, 0.05, 0.10]
minsup_table = pd.DataFrame([minsup_summary(r) for r in table_grid])
display(minsup_table.style.format({
    'relative minsup': '{:.0%}',
    'artists kept': '{:,}',
    '% of artists': '{:.1%}',
    'records kept': '{:,}',
    '% of records': '{:.1%}',
}).hide(axis='index'))

**Table 6.** Artists and user–artist records that remain when artists below each candidate *minsup* are removed. The absolute *minsup* is the relative value times $N$ = 1,892, rounded up.

In [ ]:
curve_grid = np.round(np.arange(0.005, 0.1001, 0.005), 3)
minsup_curve = pd.DataFrame([minsup_summary(r) for r in curve_grid])

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10, 4))
panels = [
    (ax_a, 'artists kept', 1, '(a) Candidate artists',
     'Artists with support ≥ minsup (log scale)'),
    (ax_b, '% of records', 100, '(b) Records covered',
     'Records involving those artists (%)'),
]
for ax, column, scale, title, ylabel in panels:
    ax.plot(minsup_curve['relative minsup'] * 100,
            minsup_curve[column] * scale, color=PRIMARY, linewidth=2)
    ax.scatter(minsup_table['relative minsup'] * 100,
               minsup_table[column] * scale, s=36, color=PRIMARY, zorder=3)
    ax.set_xlabel('Relative minsup (%)')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
ax_a.set_yscale('log')
ax_b.set_ylim(0, 100)
fig.suptitle('Raising minsup removes artists much faster than records',
             x=0.01, ha='left', fontweight='bold')
fig.tight_layout()
plt.show()

**Figure 4.** (a) Number of artists whose support is at least each relative *minsup*, on a log scale. (b) Share of the 92,834 user–artist records that involve those artists. Markers show the thresholds in Table 6.

By downward closure, an artist below *minsup* cannot belong to any frequent itemset, so Table 6 gives the number of candidate items at each threshold. Raising *minsup* from 1% to 5% cuts the candidate artists from 838 to 135, yet those 135 artists still account for 27.8% of all records, because a few popular artists appear in many transactions. A *minsup* between 2% and 5% (38 to 95 users) leaves between 135 and 432 candidate artists that together cover 28% to 47% of the records. Section 9 compares the number of frequent itemsets and the runtime within this range before fixing a value.

### 7.7 Play Counts

In [ ]:
weights = user_artists['weight']
bins = np.logspace(0, np.log10(weights.max()), 40)

fig, ax = plt.subplots()
ax.hist(weights, bins=bins, color=PRIMARY, rwidth=0.85)
ax.set_xscale('log')
ax.axvline(weights.median(), color=MUTED, linestyle='--', linewidth=1)
label_y = ax.get_ylim()[1] * 0.9
ax.annotate(f'median {weights.median():,.0f} plays',
            xy=(weights.median(), label_y),
            xytext=(weights.median() * 12, label_y), va='center',
            color=MUTED, arrowprops={'arrowstyle': '->', 'color': MUTED})
ax.set_xlabel('Play count of a user–artist record (weight, log scale)')
ax.set_ylabel('Number of records')
ax.set_title('Play counts span more than five orders of magnitude')
plt.show()

print(weights.describe().round(1).to_string())

**Figure 5.** Distribution of play counts (`weight`) across the 92,834 user–artist records, with the x-axis on a log scale. The median record has 260 plays, but play counts range from 1 to 352,698.

Play counts are heavily right-skewed, so two artists in the same top-50 list can differ in plays by several orders of magnitude. Following the class notebook, this report uses presence only: an artist belongs to $T_i$ if it is in user $i$'s top 50, whatever its play count. This keeps the analysis within standard FIM, which works on sets, and stops a few extreme listeners from dominating the results. The cost is that a barely played artist counts as much as a favorite. Section 15 discusses a play-count threshold as an alternative.

### 7.8 Tags

In [ ]:
tag_names = tags.set_index('tagID')['tagValue']
top_tags = (
    user_tagged['tagID'].value_counts().head(15)
    .rename(index=tag_names)
    .sort_values()  # ascending, so the largest bar is drawn at the top
)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(top_tags.index, top_tags.values, height=0.6, color=PRIMARY)
for tag, count in top_tags.items():
    ax.text(count + 60, tag, f'{count:,}',
            va='center', fontsize=9, color=MUTED)
ax.set_xlim(0, top_tags.max() * 1.12)
ax.grid(axis='y', visible=False)
ax.set_xlabel('Tag assignments')
ax.set_title('Broad genres dominate the most-used tags')
plt.show()

tagged_artists = user_tagged['artistID'].unique()
min_support_1pct = np.ceil(0.01 * n_users)
candidates_1pct = artist_support[artist_support >= min_support_1pct].index
print(f"Distinct tags used: {user_tagged['tagID'].nunique():,}")
print(f'Artists with at least one tag: {len(tagged_artists):,}')
print(f'Listened artists with a tag: '
      f'{artist_support.index.isin(tagged_artists).mean():.1%}')
print(f'Artists with support >= 1% that have a tag: '
      f'{candidates_1pct.isin(tagged_artists).mean():.1%}')

**Figure 6.** The 15 most-used tags, by number of tag assignments in `user_taggedartists.dat`. Broad genres such as *rock*, *pop*, and *alternative* lead.

Users tagged 12,523 artists with 9,749 distinct tags. Only 68.8% of listened artists have a tag, but every artist with a support of at least 1% has one, so tags can describe every artist that can appear in a frequent itemset. Besides broad genres (rock, pop, alternative, electronic, indie), the leading tags include descriptors such as *female vocalists* and *80s*. These hint at the taste clusters that the frequent itemsets in Section 9 may reveal.

### 7.9 Key EDA Takeaways

1. **The data needs no cleaning before mining.** `user_artists.dat` has no nulls, duplicates, or broken IDs (Table 3). The only nulls, 444 missing `pictureURL`s, belong to obscure artists in an unused column (Table 4).
2. **Transactions are top-50 lists.** 96.7% of users have exactly 50 artists (Figure 1), so the itemsets describe artists that co-occur in users' top-50 lists, not in their full histories.
3. **The data is very sparse.** Only 0.28% of the user × artist matrix is non-zero (Table 5), which suits the list-of-sets and `csr_matrix` representations used in class.
4. **Artist support is extremely long-tailed.** 60.6% of artists appear in one transaction and only 4.8% reach 1% support (Figure 2, Table 6), so association rules can only ever cover popular artists.
5. **A *minsup* of 2% to 5% is the range to test.** It keeps 135 to 432 candidate artists covering 28% to 47% of records (Table 6, Figure 4).
6. **Popular artists will dominate the rules.** The most common artists are mainstream pop acts present in up to 32.3% of transactions (Figure 3), so rules must be ranked by lift and not by confidence alone.
7. **Play counts are skewed and are ignored by design.** Presence in the top 50 defines a transaction, in line with the class approach (Figure 5).

## 8. Preprocessing: Building the Transaction Database

This section turns the user–artist records into the transaction database $\mathscr{T}$, using the representation from class: a list of sets, where each set $T_i$ holds the names of the artists in user $i$'s top 50. Only presence is kept, so `weight` is dropped (Section 7.7). The section then builds the equivalent sparse binary matrix and splits users into training, validation, and test sets for the evaluation in Section 12.

### 8.1 Transaction Database

In [ ]:
SEED = 42  # fixed seed for every random step in the report

# Exact artist names are unique (Table 3), so names can label the items
assert artist_names.is_unique

transactions = (
    user_artists.assign(artist=user_artists['artistID'].map(artist_names))
    .groupby('userID')['artist'].agg(set)
    .sort_index()
)
db = transactions.tolist()
user_ids = transactions.index.to_numpy()

print(f'Transactions (N): {len(db):,}')
print(f'Distinct items (d): {len(set().union(*db)):,}')
print(f'Items across all transactions: {sum(len(t) for t in db):,}')

sample = transactions.sample(3, random_state=SEED)
display(pd.DataFrame({
    'n_i': sample.map(len),
    'first six artists (alphabetical)': sample.map(
        lambda t: ', '.join(sorted(t)[:6]) + ', ...'),
}))

**Table 7.** Three randomly chosen transactions, each with its length $n_i$ and its first six artists in alphabetical order.

The database holds $N$ = 1,892 transactions over $d$ = 17,632 distinct artists, with 92,834 items in total, matching the record count in Table 5. No user, artist, or record is removed when building it. Section 7.1 found nothing to clean, and dropping artists below *minsup* is unnecessary because Apriori and FP-Growth already ignore them (downward closure). Keeping every artist also keeps the evaluation honest: hidden artists that no rule can reach still count against recall (Section 8.4).

### 8.2 Sparse Matrix Representation

In [ ]:
def to_sparse(db):
    """Convert a list-of-sets transaction database to a binary CSR matrix.

    Parameters
    ----------
    db : list of set
        Transactions; each set holds the items in one transaction.

    Returns
    -------
    matrix : scipy.sparse.csr_matrix
        N x d matrix with a 1 where item j is in transaction i.
    items : list of str
        Item label of each column, sorted alphabetically.
    """
    items = sorted(set().union(*db))
    column_of = {item: j for j, item in enumerate(items)}
    rows, cols = [], []
    for i, transaction in enumerate(db):
        for item in transaction:
            rows.append(i)
            cols.append(column_of[item])
    data = np.ones(len(rows), dtype=np.int8)
    matrix = csr_matrix((data, (rows, cols)), shape=(len(db), len(items)))
    return matrix, items


db_matrix, db_items = to_sparse(db)

# Column sums of the matrix must equal the supports counted in Section 7
matrix_support = pd.Series(
    np.asarray(db_matrix.sum(axis=0)).ravel(), index=db_items)
expected_support = artist_support.rename(index=artist_names)
assert (matrix_support == expected_support[db_items]).all()

print(f'Matrix shape (N x d): {db_matrix.shape[0]:,} x '
      f'{db_matrix.shape[1]:,}')
print(f'Non-zero cells: {db_matrix.nnz:,}')
print(f'Density: {db_matrix.nnz / np.prod(db_matrix.shape):.2%}')
print('First five columns:', db_items[:5])

`to_sparse` follows the approach of the class notebook: one row per transaction and one column per artist, with the columns sorted alphabetically. The resulting 1,892 × 17,632 matrix has 92,834 non-zero cells (density 0.28%), and its column sums equal the artist supports from Section 7. This confirms that the list of sets and the matrix describe the same data. The list of sets is the input to pyFIM in Section 9, while the matrix allows fast column-wise support counts.

### 8.3 Training, Validation, and Test Split

The recommender is evaluated by hiding part of each evaluation user's transaction and checking whether it recommends the hidden artists. The protocol has four rules:

1. **Only users with at least 10 artists are evaluated.** Hiding 20% of a shorter transaction would leave one hidden artist or none, which is too little ground truth to score. Users below this threshold are not discarded; their full transactions go to the training set.
2. **Eligible users are split at random into 70% training, 15% validation, and 15% test**, with a fixed seed for reproducibility. A 70% training share keeps as many transactions as possible for mining, while 15% still leaves over 250 users in each evaluation set.
3. **For each validation and test user, 20% of their artists are hidden at random** (10 of 50 for a full transaction). The remaining *visible* artists are the recommender's input and the *hidden* artists are the ground truth. Ten hidden artists per user are enough to measure precision and recall up to $k$ = 10, while 40 visible artists leave plenty of antecedents to match.
4. **Itemsets and rules are mined on the training transactions only.** No artist of a validation or test user, visible or hidden, can influence the rules, so there is no leakage. The validation set is used to tune *minsup*, *minconf*, $k$, and the scoring function in Section 12; the test set is used once, for the final metrics.

Because only training transactions are mined, every relative *minsup* in Sections 9–10 is converted to an absolute count using the number of training transactions, not $N$ = 1,892.

In [ ]:
MIN_ARTISTS_EVAL = 10  # shorter transactions stay in training only
HIDE_FRACTION = 0.2    # share of each evaluation user's artists hidden
VAL_SHARE = 0.15
TEST_SHARE = 0.15

rng = np.random.default_rng(SEED)
basket_sizes = transactions.map(len)
eligible = basket_sizes[basket_sizes >= MIN_ARTISTS_EVAL].index.to_numpy()

shuffled = rng.permutation(eligible)
n_test = round(TEST_SHARE * len(eligible))
n_val = round(VAL_SHARE * len(eligible))
test_users = np.sort(shuffled[:n_test])
val_users = np.sort(shuffled[n_test:n_test + n_val])
train_users = np.setdiff1d(user_ids, np.concatenate([test_users, val_users]))


def hide_items(transactions, users, rng, fraction=HIDE_FRACTION):
    """Split each user's transaction into visible and hidden artists.

    Parameters
    ----------
    transactions : pandas.Series
        Set of artist names per user, indexed by userID.
    users : array-like of int
        Users whose transactions are split.
    rng : numpy.random.Generator
        Random generator, so the split is reproducible.
    fraction : float, optional
        Share of each transaction to hide (at least one artist).

    Returns
    -------
    visible, hidden : dict of int to set
        Visible (input) and hidden (ground truth) artists per user.
    """
    visible, hidden = {}, {}
    for user in users:
        items = sorted(transactions[user])  # fixed order for the draw
        n_hide = max(1, round(fraction * len(items)))
        hide_idx = rng.choice(len(items), size=n_hide, replace=False)
        hidden[user] = {items[j] for j in hide_idx}
        visible[user] = set(items) - hidden[user]
    return visible, hidden


db_train = [transactions[user] for user in train_users]
val_visible, val_hidden = hide_items(transactions, val_users, rng)
test_visible, test_hidden = hide_items(transactions, test_users, rng)

# Sanity checks: the user sets are disjoint and every split is complete
assert not set(train_users) & (set(val_users) | set(test_users))
assert not set(val_users) & set(test_users)
assert len(train_users) + len(val_users) + len(test_users) == len(db)
assert all(val_visible[u] | val_hidden[u] == transactions[u]
           for u in val_users)
assert all(not test_visible[u] & test_hidden[u] for u in test_users)

In [ ]:
def count_records(user_sets):
    """Total number of user-artist records in a collection of sets."""
    return sum(len(items) for items in user_sets)


short_users = basket_sizes[basket_sizes < MIN_ARTISTS_EVAL].index
steps = [
    ('All users', user_ids, count_records(db)),
    (f'Fewer than {MIN_ARTISTS_EVAL} artists (training only)',
     short_users, count_records(transactions[short_users])),
    ('Training set', train_users, count_records(db_train)),
    ('Validation set', val_users,
     count_records(transactions[val_users])),
    ('   visible (input)', val_users, count_records(val_visible.values())),
    ('   hidden (ground truth)', val_users,
     count_records(val_hidden.values())),
    ('Test set', test_users, count_records(transactions[test_users])),
    ('   visible (input)', test_users,
     count_records(test_visible.values())),
    ('   hidden (ground truth)', test_users,
     count_records(test_hidden.values())),
]
split_table = pd.DataFrame(
    [(name, len(users), records) for name, users, records in steps],
    columns=['subset', 'users', 'records'],
)
split_table['% of users'] = split_table['users'] / len(db)
split_table['% of records'] = split_table['records'] / n_records
display(split_table.style.format({
    'users': '{:,}',
    'records': '{:,}',
    '% of users': '{:.1%}',
    '% of records': '{:.1%}',
}).hide(axis='index'))

**Table 8.** Users and user–artist records at each step of the split. Percentages are relative to all 1,892 users and 92,834 records. The users with fewer than 10 artists are part of the training set, and the visible and hidden rows divide their evaluation set.

Only 18 users (1.0%) have fewer than 10 artists. Together they hold 54 records (0.1%), and all of them join the training set. The remaining 1,874 eligible users split into 1,312 training, 281 validation, and 281 test users, so the training set has 1,330 transactions holding 64,937 records (69.9%). After hiding, each evaluation set has about 11,100 to 11,200 visible records and about 2,800 hidden ones, or close to 10 hidden artists per user (2,779 across the 281 test users). No record is discarded at any step; every record belongs to exactly one subset.

### 8.4 How Much of the Ground Truth Can Rules Reach?

An association rule can only recommend an artist that belongs to a frequent itemset in the training data, which requires the artist's own training support to reach *minsup* (downward closure). Table 9 therefore gives an upper bound on the share of hidden test artists that any rule-based recommender could find at each candidate *minsup*.

In [ ]:
n_train = len(db_train)
train_support = pd.Series(
    [artist for transaction in db_train for artist in transaction]
).value_counts()
hidden_test = pd.Series(
    [artist for items in test_hidden.values() for artist in items])
in_training = hidden_test.isin(train_support.index)
print(f'Training transactions: {n_train:,}')
print(f'Distinct artists in training: {len(train_support):,} '
      f'({len(train_support) / n_artists:.1%} of all artists)')
print(f'Hidden test records whose artist never appears in training: '
      f'{(~in_training).sum():,} ({(~in_training).mean():.1%})')

reach_rows = []
for rel_minsup in table_grid:
    abs_minsup = int(np.ceil(rel_minsup * n_train))
    frequent = train_support.index[train_support >= abs_minsup]
    reach_rows.append({
        'relative minsup': rel_minsup,
        'absolute minsup (training)': abs_minsup,
        'frequent artists': len(frequent),
        'reachable hidden test records': hidden_test.isin(frequent).mean(),
    })
display(pd.DataFrame(reach_rows).style.format({
    'relative minsup': '{:.0%}',
    'reachable hidden test records': '{:.1%}',
}).hide(axis='index'))

**Table 9.** For each candidate *minsup*, the absolute threshold on the training transactions, the number of artists that meet it, and the share of hidden test records whose artist meets it. This share is the highest recall that a rule-based recommender could reach.

Of the 17,632 artists, 14,260 (80.9%) appear in at least one training transaction. 372 hidden test records (13.4%) belong to artists that never appear in training, so no method trained on these users could recommend them. *minsup* is a much tighter limit: at 1%, only 55.6% of hidden test records are reachable, and at 5%, only 26.2%. Within the 2% to 5% range examined in Section 9, recall therefore cannot exceed 26% to 45%. This is a direct cost of the long tail in Figure 2, so Section 12 judges the recommender's recall against this ceiling rather than against 100%.

## 9. Frequent Itemset Mining

**GUIDE**
- Run **Apriori and FP-Growth** (`fim.apriori`, `fim.fpgrowth`) on the training database.
  - pyFIM note from class: **`supp` > 0 is a percentage; `supp` < 0 is absolute support**. For example, `supp=-57` means at least 57 users (~3%).
  - `zmin=2` keeps itemsets of 2+ artists; `zmax` caps the itemset size.
- **Choosing minsup:** try several values and tabulate **# frequent itemsets and runtime for each algorithm**. Reference counts on the full data: 5% → 8,435 itemsets, 3% → 66,579, 2% → 324,088. The count explodes quickly because pop artists co-occur densely. Pick a value that is meaningful and manageable, and justify it.
- **Apriori vs FP-Growth:**
  - Confirm both return **the same itemsets**.
  - Compare runtime with a table or bar chart, and link it to the class discussion (FP-Growth avoids candidate generation, which matters when there are many items).
- **Validate your class implementation (optional but strong):** run your own `apriori`/`fpgrowth` from the class exercises on a **small subset** (e.g., top-30 artists, or a high minsup) and assert they match pyFIM. They are pure-Python and will be too slow on the full data, so say this.
- Report:
  - # frequent itemsets by size $k$
  - # **maximal** (`target='m'`) and **closed** (`target='c'`) itemsets, and why they compactly summarize the result
  - Top frequent $k$-itemsets ($k \geq 2$) with absolute and relative support
- **Interpret:** do the itemsets form recognizable taste clusters (pop divas, 80s synth-pop, metal…)? Tags can help label them.

In [ ]:
# TODO: grid over minsup -> counts & runtime for fim.apriori vs fim.fpgrowth
# e.g. t=time.time(); fi = fim.fpgrowth(db_train, supp=-57, zmin=1); time.time()-t

In [ ]:
# TODO: final itemsets at chosen minsup; counts by size; maximal (target='m') & closed (target='c')

In [ ]:
# TODO (optional): check your own class functions match pyFIM on a small subset

## 10. Association Pattern Mining

**GUIDE**
- Generate rules with pyFIM, e.g. `fim.fpgrowth(db_train, target='r', supp=..., conf=..., zmin=2, report='acl')`
  - `report` codes: `a` = absolute support, `s` = relative support, `c` = confidence, `l` = lift.
  - ⚠️ pyFIM returns rules as **`(consequent, antecedent, ...)`**, consequent first. Convert to a DataFrame with explicit `antecedent` / `consequent` columns, like the class `assoc` output.
  - Or use your own `assoc(fi, minconf, db_size)` from class on the frequent itemsets, and cross-check it with pyFIM.
- **Choose and justify `minconf`.** Expect huge rule counts: on the full data, minsup 3% with minconf 50% gives over 1 million rules. Control this with a higher minsup, `zmax`, a lift threshold, or by keeping only single-item consequents.
- Present **top rules by lift and by confidence**, sorted like the class `assoc` (lift → confidence → support). Explain a few in plain English, e.g. *"users who listen to Dave Gahan and Depeche Mode also tend to listen to Martin L. Gore (conf 0.62, lift ≈ 28)"*.
- Discuss **popularity bias:** rules ending in mainstream artists (e.g., Lady Gaga) can have high confidence but lift near 1, so explain why lift matters for recommendations.
- Visualize: support vs. confidence scatter colored by lift; *(optional)* a network graph of top rules, or a lift heatmap of top artist pairs.

In [ ]:
# TODO: rules = fim.fpgrowth(db_train, target='r', supp=..., conf=..., zmin=2, report='acl')

In [ ]:
# TODO: to DataFrame (antecedent, consequent, support, confidence, lift); sort; filter lift > 1

In [ ]:
# TODO: plots + top-rule tables

## 11. Recommender System from Association Rules

**GUIDE**
- Explain the logic step by step:
  1. Input: the set of artists a user listens to.
  2. Select rules whose **antecedent ⊆ user's artists**.
  3. Keep consequents the user **doesn't already have**.
  4. **Score and rank** them (e.g., max confidence, max lift, or the sum of confidences over matching rules). Justify the choice.
  5. Return the top-*k*.
- Write a reusable function (it can go in a supplementary `.py` file; reference it here).
- Edge case: users with no matching rule fall back to the **most popular unheard artists**. Report how often this happens.
- **Demo:** 2–3 anonymous test users showing their input artists, the recommendations, and the rule that triggered each one (explainability is a key selling point).

In [ ]:
def recommend(user_items, rules_df, k=10):
    """TODO: top-k artists from rules whose antecedent is a subset of user_items."""
    pass

In [ ]:
# TODO: demo for a few test users

## 12. Performance Evaluation

**GUIDE**: ⚠️ **Required by the lab** ("Recommender systems should have a performance evaluation").
- Restate the protocol from Section 8: rules come from training users; for each test user, the visible artists are the input and the hidden artists are the ground truth.
- Metrics:
  - **Precision@k**, **Recall@k**, **Hit rate@k** (≥ 1 hidden artist recommended), optionally F1@k
  - **Coverage:** % of test users getting at least one rule-based recommendation; % of artists ever recommended
- **Baseline:** a popularity recommender (top-k most popular artists the user hasn't heard). The rule-based recommender should beat it, so state whether it does.
- *(Optional)* Sensitivity of the metrics to minsup, minconf, and k (line plots).
- Show a results table and chart, then interpret: where does it work well (mainstream tastes) and where does it fail (niche users or artists below minsup)?

In [ ]:
# TODO: evaluation loop -> precision@k, recall@k, hit rate@k, coverage

In [ ]:
# TODO: popularity baseline + comparison table/plot

In [ ]:
# TODO (optional): sensitivity analysis

## 13. Results and Discussion

**GUIDE**
- Synthesize what the results **mean**; don't just repeat tables.
- Answer each sub-question from the Problem Statement explicitly.
- Discuss the algorithm comparison (Apriori vs FP-Growth runtime on this data) and the effect of the minsup/minconf choices.
- **Limitations:**
  - Top-50 cap per user, and presence only (play counts ignored)
  - Popularity bias, so the long tail of niche artists never reaches minsup
  - Small, 2011-era user base, so findings may not generalize
  - Results are sensitive to thresholds

## 14. Conclusion

**GUIDE**
- 1–2 paragraphs: problem, approach, key findings with numbers, and a clear verdict on whether association rules are a viable basis for artist recommendations on this data.

## 15. Recommendations

**GUIDE**: Give actionable suggestions, like Problem 10 in the class notebook (three concrete suggestions backed by specific itemsets or rules with their numbers).
- **For the platform / stakeholders**, e.g.:
  - Use high-lift rules for "Fans also like" or auto-playlists
  - Bundle or co-promote artists with strong associations
  - Use the rules to seed new users (cold start) from one or two chosen artists
- **Future work**, e.g.:
  - Use `weight` (count an artist only above a play threshold, or weight the rules)
  - Compare against collaborative filtering (item-based or matrix factorization)
  - Mine **tag/genre** itemsets to cover niche artists
  - Use `user_friends.dat` for social recommendations

## 16. References

**GUIDE**: Use one consistent citation style.
- Cantador, I., Brusilovsky, P., & Kuflik, T. (2011). 2nd Workshop on Information Heterogeneity and Fusion in Recommender Systems (HetRec 2011). *Proc. 5th ACM Conference on Recommender Systems (RecSys 2011)*. ACM.
- Last.fm, http://www.lastfm.com · GroupLens HetRec 2011, https://grouplens.org/datasets/hetrec-2011/
- Aggarwal, C. (2015). *Data Mining: The Textbook*. Springer.
- Fournier-Viger, P., Lin, J. C. W., Vo, B., Chi, T. T., Zhang, J., & Le, H. B. (2017). A survey of itemset mining. *WIREs Data Mining and Knowledge Discovery, 7*(4), e1207.
- Borgelt, C. *PyFIM: Frequent Item Set Mining for Python*. http://www.borgelt.net/pyfim.html
- Agrawal, R., & Srikant, R. (1994). Fast algorithms for mining association rules. *Proc. VLDB.* (Apriori)
- Han, J., Pei, J., & Yin, Y. (2000). Mining frequent patterns without candidate generation. *Proc. ACM SIGMOD.* (FP-Growth)
- Course materials: *Frequent Itemset Mining* notebook (C. Alis) and BDCC Sessions 3–4 slides.
- GroupLens. (2013). HetRec 2011: Last.fm dataset. https://grouplens.org/datasets/hetrec-2011/
- Talwar, K. S., Oraganti, A., Mahajan, N., & Narsale, P. (n.d.). Recommendation system using Apriori algorithm.